# Student Pass/Fail Prediction Training Pipeline
This notebook demonstrates the object-oriented approach to loading, cleaning, training, and clustering the 9-feature student dataset using the `src/` modules.

In [1]:
from dotenv import load_dotenv
load_dotenv()

import os
import sys
sys.path.append(os.path.abspath('..'))

from src.data_processor import DataProcessor
from src.model_trainer import ModelTrainer

### 1. Data Cleaning and Preprocessing

In [2]:
processor = DataProcessor()

# 1. Load Data
df = processor.load_data()
print(f"Raw dataset shape: {df.shape}")

# 2. Clean Data
df = processor.fill_nulls(df)
df_clean = processor.remove_outliers(df)
print(f"After IQR Outlier removal: {df_clean.shape}")

# 3. Preprocess and Scale
X_scaled, y, df_clean = processor.preprocess(df_clean)

# 4. Split and SMOTE
X_train_sm, X_test, y_train_sm, y_test = processor.split_and_balance(X_scaled, y)
print(f"SMOTE Train Set Shape: {X_train_sm.shape} | Test Set Shape: {X_test.shape}")

Loaded: 200,000 rows | Columns: ['gender', 'age', 'parental_education_level', 'family_income', 'daily_study_hours', 'attendance_rate', 'sleep_hours', 'stress_level', 'motivation_score', 'private_tutoring', 'internet_quality', 'math_score', 'reading_score', 'writing_score', 'pass_fail']
Raw dataset shape: (200000, 15)
Null values filled with column medians.
After IQR outlier removal: 190,523 rows
After IQR Outlier removal: (190523, 15)
Class distribution → Pass: 103,195 (54.2%)  Fail: 87,328 (45.8%)


Train: 152,418  |  Test: 38,105


After SMOTE → Pass: 82,556  Fail: 82,556
SMOTE Train Set Shape: (165112, 9) | Test Set Shape: (38105, 9)


### 2. Model Training via GridSearchCV

In [3]:
trainer = ModelTrainer()
trainer.train_and_evaluate(X_train_sm, y_train_sm, X_test, y_test)


Model                  Accuracy    Precision     Recall F1 (weighted)
------------------------------------------------------------------

Running GridSearchCV for LogisticRegression...
Fitting 3 folds for each of 3 candidates, totalling 9 fits


Best params for LogisticRegression: {'C': 10.0}
LogisticRegression       1.0000       1.0000     1.0000       1.0000

Running GridSearchCV for XGBoost...
Fitting 3 folds for each of 8 candidates, totalling 24 fits


Best params for XGBoost: {'learning_rate': 0.1, 'max_depth': 6, 'n_estimators': 100}
XGBoost                  0.9981       0.9981     0.9981       0.9981

🏆 Best overall model: LogisticRegression  (F1 = 1.0000)
Winning hyperparameters: {'C': 10.0}

── Final Metrics (LogisticRegression) ──────────────────
  Accuracy  : 1.0000
  Precision : 1.0000
  Recall    : 1.0000
  F1        : 1.0000

── Classification Report ──────────────────────────────
              precision    recall  f1-score   support

        Fail       1.00      1.00      1.00     17466
        Pass       1.00      1.00      1.00     20639

    accuracy                           1.00     38105
   macro avg       1.00      1.00      1.00     38105
weighted avg       1.00      1.00      1.00     38105

── Confusion Matrix ───────────────────────────────────
  TN:   17,466  FP:        0
  FN:        0  TP:   20,639


### 3. KMeans Behavior Clustering

In [4]:
trainer.train_kmeans(df_clean, processor.feature_cols, processor.scaler)
print("KMeans trained with k=3")

KMeans trained with k=3


### 4. Serialize Architectures

In [5]:
trainer.save_artifacts(
    processor.scaler, processor.feature_names, processor.feature_cols
)
print("✅ Pipeline complete: Artifacts saved to models/")


✅ All artifacts saved to models/
✅ Pipeline complete: Artifacts saved to models/
